# Q1. Embedding a query

In [1]:
from embedder import Embedder

query = "How does approximate nearest neighbor search work?"

embedder = Embedder()
v = embedder.encode(query)

len(v)    # 384
v[0]      # -0.020582... → pick -0.02

np.float64(-0.020582036807885073)

# Q2. Cosine similarity

In [2]:
from gitsource import GithubRepositoryDataReader
from embedder import Embedder
import numpy as np

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

doc = next(
    d for d in documents
    if d["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
)

query = "How does approximate nearest neighbor search work?"
embedder = Embedder()

query_vec = embedder.encode(query)
doc_vec = embedder.encode(doc["content"])

np.dot(query_vec, doc_vec)   # ~0.361 → 0.37

np.float64(0.361070280302606)

# Q3. Chunking and search by hand

In [3]:
from gitsource import chunk_documents
from embedder import Embedder
import numpy as np

chunks = chunk_documents(documents, size=2000, step=1000)

query = "How does approximate nearest neighbor search work?"
embedder = Embedder()
v = embedder.encode(query)

X = embedder.encode_batch([c["content"] for c in chunks])
scores = X.dot(v)

best = int(np.argmax(scores))
chunks[best]["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

# Q4. Vector search with minsearch

In [4]:
from gitsource import chunk_documents
from minsearch import VectorSearch
from embedder import Embedder

chunks = chunk_documents(documents, size=2000, step=1000)

embedder = Embedder()
X = embedder.encode_batch([c["content"] for c in chunks])

index = VectorSearch(keyword_fields=["filename"])
index.fit(X, chunks)

query = "What metric do we use to evaluate a search engine?"
v = embedder.encode(query)

results = index.search(v, num_results=5)
results[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

# Q5. Text search vs vector search

In [5]:
from minsearch import Index, VectorSearch

text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(X, chunks)

query = "How do I store vectors in PostgreSQL?"
v = embedder.encode(query)

text_results = text_index.search(query, num_results=5)
vector_results = vector_index.search(v, num_results=5)

text_files = {r["filename"] for r in text_results}
vector_files = {r["filename"] for r in vector_results}

vector_files - text_files
# {'02-vector-search/lessons/08-pgvector.md'}

{'02-vector-search/lessons/08-pgvector.md'}

# Q6. Hybrid search

In [6]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

query = "How do I give the model access to tools?"
v = embedder.encode(query)

vector_results = vector_index.search(v, num_results=5)
text_results = text_index.search(query, num_results=5)

results = rrf([vector_results, text_results])
results[0]["filename"]

'01-agentic-rag/lessons/13-function-calling.md'